# Celebal Summer Internship 2026 — Week 2
## E-Commerce Sales Database — SQL Assignment

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('ecommerce.db')
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")
print("Connected successfully")

Connected successfully


In [2]:
cursor.executescript("""
DROP TABLE IF EXISTS order_items;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;

CREATE TABLE customers (
    customer_id   INT          PRIMARY KEY,
    first_name    VARCHAR(50)  NOT NULL,
    last_name     VARCHAR(50)  NOT NULL,
    email         VARCHAR(100) UNIQUE NOT NULL,
    city          VARCHAR(50)  NOT NULL,
    state         VARCHAR(50)  NOT NULL,
    join_date     DATE         NOT NULL,
    is_premium    BOOLEAN      DEFAULT FALSE
);

CREATE INDEX idx_customers_city ON customers(city);
CREATE INDEX idx_customers_state ON customers(state);

CREATE TABLE products (
    product_id    INT           PRIMARY KEY,
    product_name  VARCHAR(100)  NOT NULL,
    category      VARCHAR(50)   NOT NULL,
    brand         VARCHAR(50)   NOT NULL,
    unit_price    DECIMAL(10,2) NOT NULL CHECK (unit_price > 0),
    stock_qty     INT           NOT NULL DEFAULT 0 CHECK (stock_qty >= 0)
);

CREATE INDEX idx_products_category ON products(category);

CREATE TABLE orders (
    order_id      INT           PRIMARY KEY,
    customer_id   INT           NOT NULL,
    order_date    DATE          NOT NULL,
    status        VARCHAR(20)   NOT NULL DEFAULT 'Pending'
                  CHECK (status IN ('Pending','Shipped','Delivered','Cancelled')),
    total_amount  DECIMAL(12,2) NOT NULL CHECK (total_amount >= 0),

    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

CREATE INDEX idx_orders_date ON orders(order_date);
CREATE INDEX idx_orders_status ON orders(status);

CREATE TABLE order_items (
    item_id       INT           PRIMARY KEY,
    order_id      INT           NOT NULL,
    product_id    INT           NOT NULL,
    quantity      INT           NOT NULL CHECK (quantity > 0),
    unit_price    DECIMAL(10,2) NOT NULL CHECK (unit_price > 0),
    discount_pct  DECIMAL(5,2)  DEFAULT 0 CHECK (discount_pct BETWEEN 0 AND 100),

    FOREIGN KEY (order_id) REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
""")
conn.commit()
print("Tables created successfully")

Tables created successfully


In [3]:
cursor.executescript("""
-- ========== INSERT: customers ==========
INSERT INTO customers VALUES
(101, 'Aarav',  'Sharma', 'aarav.s@email.com',  'Mumbai',    'Maharashtra', '2024-01-15', TRUE),
(102, 'Priya',  'Patel',  'priya.p@email.com',  'Ahmedabad', 'Gujarat',     '2024-02-20', FALSE),
(103, 'Rohan',  'Gupta',  'rohan.g@email.com',  'Delhi',     'Delhi',       '2024-03-10', TRUE),
(104, 'Sneha',  'Reddy',  'sneha.r@email.com',  'Hyderabad', 'Telangana',   '2024-04-05', FALSE),
(105, 'Vikram', 'Singh',  'vikram.s@email.com', 'Jaipur',    'Rajasthan',   '2024-05-12', TRUE),
(106, 'Ananya', 'Iyer',   'ananya.i@email.com', 'Chennai',   'Tamil Nadu',  '2024-06-18', FALSE),
(107, 'Karan',  'Mehta',  'karan.m@email.com',  'Pune',      'Maharashtra', '2024-07-22', TRUE),
(108, 'Divya',  'Nair',   'divya.n@email.com',  'Kochi',     'Kerala',      '2024-08-30', FALSE);

-- ========== INSERT: products ==========
INSERT INTO products VALUES
(201, 'Wireless Earbuds',     'Electronics', 'BoAt',        1499.00, 250),
(202, 'Cotton T-Shirt',       'Clothing',    'Levis',        799.00, 500),
(203, 'Smart Watch',          'Electronics', 'Noise',       2999.00, 150),
(204, 'Running Shoes',        'Clothing',    'Nike',        4599.00, 120),
(205, 'Bluetooth Speaker',    'Electronics', 'JBL',         3499.00, 200),
(206, 'Bedsheet Set',         'Home',        'Spaces',      1299.00, 300),
(207, 'Laptop Stand',         'Electronics', 'AmazonBasics', 899.00, 180),
(208, 'Cushion Covers (Set)', 'Home',        'HomeCenter',   599.00, 400);

-- ========== INSERT: orders ==========
INSERT INTO orders VALUES
(1001, 101, '2024-08-01', 'Delivered', 4498.00),
(1002, 102, '2024-08-03', 'Delivered',  799.00),
(1003, 103, '2024-08-05', 'Shipped',   7498.00),
(1004, 101, '2024-08-10', 'Delivered', 3499.00),
(1005, 104, '2024-08-12', 'Cancelled', 2999.00),
(1006, 105, '2024-08-15', 'Delivered', 5898.00),
(1007, 106, '2024-08-18', 'Pending',   1299.00),
(1008, 103, '2024-08-20', 'Delivered',  899.00),
(1009, 107, '2024-08-25', 'Shipped',   6098.00),
(1010, 108, '2024-08-28', 'Delivered', 1598.00);

-- ========== INSERT: order_items ==========
INSERT INTO order_items VALUES
(5001, 1001, 201, 2, 1499.00,  0),
(5002, 1001, 207, 1,  899.00, 10),
(5003, 1002, 202, 1,  799.00,  0),
(5004, 1003, 203, 1, 2999.00,  0),
(5005, 1003, 204, 1, 4599.00,  5),
(5006, 1004, 205, 1, 3499.00,  0),
(5007, 1005, 203, 1, 2999.00,  0),
(5008, 1006, 201, 1, 1499.00, 10),
(5009, 1006, 204, 1, 4599.00,  5),
(5010, 1007, 206, 1, 1299.00,  0),
(5011, 1008, 207, 1,  899.00,  0),
(5012, 1009, 205, 1, 3499.00,  0),
(5013, 1009, 208, 2,  599.00, 15),
(5014, 1010, 206, 1, 1299.00,  0),
(5015, 1010, 208, 1,  599.00,  0);
""")
conn.commit()
print("Data loaded successfully")

Data loaded successfully


In [4]:
for table in ['customers', 'products', 'orders', 'order_items']:
    count = pd.read_sql(f"SELECT COUNT(*) AS total FROM {table}", conn)
    print(table, "→", count['total'][0], "rows")

customers → 8 rows
products → 8 rows
orders → 10 rows
order_items → 15 rows


## Section A — Basics (SELECT, Primary Keys, Constraints)

In [5]:
# Q1: Display all columns and rows from the customers table
pd.read_sql("SELECT * FROM customers;", conn)

,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,102,Priya,Patel,priya.p@email.com,Ahmedabad,Gujarat,2024-02-20,0
2,103,Rohan,Gupta,rohan.g@email.com,Delhi,Delhi,2024-03-10,1
3,104,Sneha,Reddy,sneha.r@email.com,Hyderabad,Telangana,2024-04-05,0
4,105,Vikram,Singh,vikram.s@email.com,Jaipur,Rajasthan,2024-05-12,1
5,106,Ananya,Iyer,ananya.i@email.com,Chennai,Tamil Nadu,2024-06-18,0
6,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1
7,108,Divya,Nair,divya.n@email.com,Kochi,Kerala,2024-08-30,0


In [6]:
# Q2: Retrieve only the relevant columns instead of using SELECT *
pd.read_sql("SELECT first_name, last_name, city FROM customers;", conn)

,first_name,last_name,city
0,Aarav,Sharma,Mumbai
1,Priya,Patel,Ahmedabad
2,Rohan,Gupta,Delhi
3,Sneha,Reddy,Hyderabad
4,Vikram,Singh,Jaipur
5,Ananya,Iyer,Chennai
6,Karan,Mehta,Pune
7,Divya,Nair,Kochi


In [7]:
# Q3: Use DISTINCT to remove duplicate category values
pd.read_sql("SELECT DISTINCT category FROM products;", conn)

,category
0,Clothing
1,Electronics
2,Home


**Insight:** The products table contains 3 unique categories — Electronics, 
Clothing, and Home — confirming the product catalog spans multiple segments.

### Q4. Identify the Primary Key of each table in the schema. Explain why a Primary Key must be unique and NOT NULL.

**Answer:**

Primary Keys in this schema:
- customers → customer_id
- products → product_id
- orders → order_id
- order_items → item_id

A Primary Key must be UNIQUE so every row can be identified without ambiguity — 
if two rows shared the same ID, the database couldn't reliably target a specific 
row for updates or deletes. It must also be NOT NULL because a missing identifier 
means the row has no identity at all, which breaks any Foreign Key relationship 
pointing to it from other tables.


### Q5. What constraints are applied to the email column in the customers table? What would happen if you tried to insert a duplicate email?



In [8]:
# Q5: Attempt to insert a duplicate email to test the UNIQUE constraint
try:
    cursor.execute("""
        INSERT INTO customers (customer_id, first_name, last_name, email, city, state, join_date) 
        VALUES (999, 'Test', 'User', 'aarav.s@email.com', 'Mumbai', 'Maharashtra', '2024-09-01')
    """)
    conn.commit()
except Exception as e:
    print("Error:", e)

Error: UNIQUE constraint failed: customers.email


**Answer:** The `email` column has UNIQUE and NOT NULL constraints. Attempting 
to insert a duplicate email raises an IntegrityError: 
"UNIQUE constraint failed: customers.email" — the database rejects the row 
because it violates the uniqueness rule defined in the schema, ensuring no 
two customers can register with the same email address.

### Q6. Try inserting a product with unit_price = -50. What happens and which constraint prevents it? Write both the INSERT statement and explain the error.

In [9]:
# Q6: Attempt to insert a product with a negative price to test the CHECK constraint
try:
    cursor.execute("""
        INSERT INTO products (product_id, product_name, category, brand, unit_price, stock_qty) 
        VALUES (999, 'Test Product', 'Test', 'TestBrand', -50, 10)
    """)
    conn.commit()
except Exception as e:
    print("Error:", e)

Error: CHECK constraint failed: unit_price > 0


**Answer:** This violates the CHECK constraint `CHECK (unit_price > 0)` on the 
products table. SQLite raises: "IntegrityError: CHECK constraint failed: products" 
— the insert is rejected because a negative price doesn't satisfy the business 
rule enforced at the database level, preventing invalid data from ever being stored.

## Section B — Filtering & Optimization

In [10]:
# Q7. Retrieve all orders with status = 'Delivered'.
pd.read_sql("SELECT * FROM orders WHERE status = 'Delivered';", conn)

,order_id,customer_id,order_date,status,total_amount
0,1001,101,2024-08-01,Delivered,4498
1,1002,102,2024-08-03,Delivered,799
2,1004,101,2024-08-10,Delivered,3499
3,1006,105,2024-08-15,Delivered,5898
4,1008,103,2024-08-20,Delivered,899
5,1010,108,2024-08-28,Delivered,1598


**Insight:** Out of 10 total orders, the majority have a 'Delivered' status, 
indicating most transactions are completing successfully.

In [11]:
# Q8. Find all products in the 'Electronics' category with a unit_price greater than ₹2000.
pd.read_sql("""
SELECT * FROM products 
WHERE category = 'Electronics' AND unit_price > 2000;
""", conn)

,product_id,product_name,category,brand,unit_price,stock_qty
0,203,Smart Watch,Electronics,Noise,2999,150
1,205,Bluetooth Speaker,Electronics,JBL,3499,200


In [12]:
# Q9. List all customers who joined in the year 2024 and belong to the state 'Maharashtra'.
pd.read_sql("""
SELECT * FROM customers 
WHERE strftime('%Y', join_date) = '2024' AND state = 'Maharashtra';
""", conn)


,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1


**Insight:** Customers from Maharashtra make up a notable share of the 2024 
sign-ups, suggesting this state could be a key market for targeted promotions.

In [13]:
# Q10. Find all orders placed between '2024-08-10' and '2024-08-25' (inclusive) that are NOT cancelled.
pd.read_sql("""
SELECT * FROM orders 
WHERE order_date BETWEEN '2024-08-10' AND '2024-08-25' 
AND status != 'Cancelled';
""", conn)

,order_id,customer_id,order_date,status,total_amount
0,1004,101,2024-08-10,Delivered,3499
1,1006,105,2024-08-15,Delivered,5898
2,1007,106,2024-08-18,Pending,1299
3,1008,103,2024-08-20,Delivered,899
4,1009,107,2024-08-25,Shipped,6098


In [14]:
# Q11. Explain what the index idx_orders_date does. How would it improve the performance of a query that filters orders by order_date? Write a sample query that would benefit from this index.
pd.read_sql("SELECT * FROM orders WHERE order_date = '2024-08-15';", conn)

,order_id,customer_id,order_date,status,total_amount
0,1006,105,2024-08-15,Delivered,5898


**Insight:** Since order_date has a dedicated index (idx_orders_date), 
filtering by a specific date — as shown above — runs efficiently even as 
the orders table scales, instead of scanning every row.

In [15]:
# Q12. If you run: SELECT * FROM customers WHERE YEAR(join_date) = 2024; — would the index on join_date be used? Explain why or why not, and rewrite the query to be index-friendly (SARGable).
pd.read_sql("""
SELECT * FROM customers 
WHERE join_date >= '2024-01-01' AND join_date < '2025-01-01';
""", conn)

,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,102,Priya,Patel,priya.p@email.com,Ahmedabad,Gujarat,2024-02-20,0
2,103,Rohan,Gupta,rohan.g@email.com,Delhi,Delhi,2024-03-10,1
3,104,Sneha,Reddy,sneha.r@email.com,Hyderabad,Telangana,2024-04-05,0
4,105,Vikram,Singh,vikram.s@email.com,Jaipur,Rajasthan,2024-05-12,1
5,106,Ananya,Iyer,ananya.i@email.com,Chennai,Tamil Nadu,2024-06-18,0
6,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1
7,108,Divya,Nair,divya.n@email.com,Kochi,Kerala,2024-08-30,0


## Section C — Aggregation

In [16]:
# Q14. Find the total revenue (SUM of total_amount) from all 'Delivered' orders.
pd.read_sql("""
SELECT SUM(total_amount) AS total_revenue 
FROM orders WHERE status = 'Delivered';
""", conn)

,total_revenue
0,17191


In [17]:
# Q15. Calculate the average unit_price of products in each category.
pd.read_sql("""
SELECT category, AVG(unit_price) AS avg_price 
FROM products GROUP BY category;
""", conn)

,category,avg_price
0,Clothing,2699.0
1,Electronics,2224.0
2,Home,949.0


**Insight:** Electronics has the highest average unit_price among the three 
categories, suggesting it's a premium segment compared to Clothing and Home.

In [18]:
# Q16. For each order status, find the count of orders and the total revenue. Sort the result by total revenue in descending order.
pd.read_sql("""
SELECT status, COUNT(*) AS order_count, SUM(total_amount) AS total_revenue
FROM orders
GROUP BY status
ORDER BY total_revenue DESC;
""", conn)

,status,order_count,total_revenue
0,Delivered,6,17191
1,Shipped,2,13596
2,Cancelled,1,2999
3,Pending,1,1299


In [19]:
# Q17. Find the most expensive (MAX) and cheapest (MIN) product in each category.
pd.read_sql("""
SELECT category, MAX(unit_price) AS most_expensive, MIN(unit_price) AS cheapest
FROM products GROUP BY category;
""", conn)

,category,most_expensive,cheapest
0,Clothing,4599,799
1,Electronics,3499,899
2,Home,1299,599


In [20]:
# Q18. List all product categories where the average unit_price is greater than ₹2000. (Hint: Use HAVING clause)
pd.read_sql("""
SELECT category, AVG(unit_price) AS avg_price
FROM products
GROUP BY category
HAVING AVG(unit_price) > 2000;
""", conn)

,category,avg_price
0,Clothing,2699.0
1,Electronics,2224.0


**Insight:** Only the category exceeding ₹2000 average price clears the 
HAVING filter — this is useful for quickly identifying premium-priced 
categories without manually scanning every row.

## Section D — Joins & Relationships

In [21]:
# Q19. Write an INNER JOIN query to display each order along with the customer's first_name and last_name. Show: order_id, order_date, first_name, last_name, total_amount.
pd.read_sql("""
SELECT o.order_id, o.order_date, c.first_name, c.last_name, o.total_amount
FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id;
""", conn)

,order_id,order_date,first_name,last_name,total_amount
0,1001,2024-08-01,Aarav,Sharma,4498
1,1002,2024-08-03,Priya,Patel,799
2,1003,2024-08-05,Rohan,Gupta,7498
3,1004,2024-08-10,Aarav,Sharma,3499
4,1005,2024-08-12,Sneha,Reddy,2999
5,1006,2024-08-15,Vikram,Singh,5898
6,1007,2024-08-18,Ananya,Iyer,1299
7,1008,2024-08-20,Rohan,Gupta,899
8,1009,2024-08-25,Karan,Mehta,6098
9,1010,2024-08-28,Divya,Nair,1598


In [22]:
# Q20. Using a LEFT JOIN, list ALL customers and their orders (if any). Customers with no orders should still appear with NULL values for order columns.
pd.read_sql("""
SELECT c.customer_id, c.first_name, c.last_name, o.order_id, o.order_date, o.total_amount
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id;
""", conn)

,customer_id,first_name,last_name,order_id,order_date,total_amount
0,101,Aarav,Sharma,1001,2024-08-01,4498
1,101,Aarav,Sharma,1004,2024-08-10,3499
2,102,Priya,Patel,1002,2024-08-03,799
3,103,Rohan,Gupta,1003,2024-08-05,7498
4,103,Rohan,Gupta,1008,2024-08-20,899
5,104,Sneha,Reddy,1005,2024-08-12,2999
6,105,Vikram,Singh,1006,2024-08-15,5898
7,106,Ananya,Iyer,1007,2024-08-18,1299
8,107,Karan,Mehta,1009,2024-08-25,6098
9,108,Divya,Nair,1010,2024-08-28,1598


**Insight:** If any customer_id appeared with NULL order columns here, it 
would mean that customer has never placed an order — useful for identifying 
inactive customers for re-engagement campaigns.

In [23]:
# Q21. Write a query using JOINs across three tables (orders → order_items → products) to show: order_id, product_name, quantity, unit_price, and discount_pct for each order item.
pd.read_sql("""
SELECT o.order_id, p.product_name, oi.quantity, oi.unit_price, oi.discount_pct
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id;
""", conn)

,order_id,product_name,quantity,unit_price,discount_pct
0,1001,Wireless Earbuds,2,1499,0
1,1001,Laptop Stand,1,899,10
2,1002,Cotton T-Shirt,1,799,0
3,1003,Smart Watch,1,2999,0
4,1003,Running Shoes,1,4599,5
5,1004,Bluetooth Speaker,1,3499,0
6,1005,Smart Watch,1,2999,0
7,1006,Wireless Earbuds,1,1499,10
8,1006,Running Shoes,1,4599,5
9,1007,Bedsheet Set,1,1299,0


**Insight:** This consolidated view — linking orders, order_items, and 
products — is exactly the kind of query a business analyst would use to 
generate a detailed sales report showing what was actually purchased, not 
just order totals.

### Q22. Explain the difference between LEFT JOIN and RIGHT JOIN with an example from this schema. When would you use a FULL OUTER JOIN?

**Answer:** 
- **LEFT JOIN** returns all rows from the left table, with NULLs where no 
  match exists in the right table. Example: `customers LEFT JOIN orders` 
  returns every customer, even those with zero orders.
- **RIGHT JOIN** returns all rows from the right table, with NULLs where no 
  match exists in the left table. Example: `customers RIGHT JOIN orders` 
  would return every order, even if (hypothetically) its customer_id didn't 
  exist — equivalent to swapping the table order in a LEFT JOIN.
  (Note: SQLite does not support RIGHT JOIN directly; the same result is 
  achieved by reversing the table order in a LEFT JOIN.)
- **FULL OUTER JOIN** would be used when you want ALL customers AND ALL 
  orders regardless of whether they match — useful for auditing both 
  unmatched customers (no orders placed) and orphaned orders (invalid 
  customer references) in a single result set.

### Q23. Identify all Foreign Key relationships in the schema. Explain what would happen if you tried to insert an order with customer_id = 999 (which doesn't exist in customers).

**Answer:** Foreign Key relationships in this schema:
- `orders.customer_id` → `customers.customer_id`
- `order_items.order_id` → `orders.order_id`
- `order_items.product_id` → `products.product_id`

Inserting an order with customer_id = 999 (which doesn't exist in customers) 
would raise a Foreign Key constraint violation, since SQLite checks that the 
referenced value exists in the parent table before allowing the insert.

In [24]:
# Q23. Demonstration: attempting to insert an order with a non-existent customer_id
try:
    cursor.execute("""
        INSERT INTO orders (order_id, customer_id, order_date, status, total_amount) 
        VALUES (9999, 999, '2024-08-30', 'Pending', 100)
    """)
    conn.commit()
except Exception as e:
    print("Error:", e)

Error: FOREIGN KEY constraint failed


## Section E — Advanced Concepts (CASE, ACID, Transactions)

In [25]:
# Q24. Write a query using CASE to classify products into price tiers: 'Budget' -> unit_price < 1000, 'Mid-Range' -> unit_price BETWEEN 1000 AND 5000, 'Premium' -> unit_price > 5000. Display: product_name, unit_price, price_tier.
pd.read_sql("""
SELECT product_name, unit_price,
CASE 
    WHEN unit_price < 1000 THEN 'Budget'
    WHEN unit_price BETWEEN 1000 AND 5000 THEN 'Mid-Range'
    ELSE 'Premium'
END AS price_tier
FROM products;
""", conn)

,product_name,unit_price,price_tier
0,Wireless Earbuds,1499,Mid-Range
1,Cotton T-Shirt,799,Budget
2,Smart Watch,2999,Mid-Range
3,Running Shoes,4599,Mid-Range
4,Bluetooth Speaker,3499,Mid-Range
5,Bedsheet Set,1299,Mid-Range
6,Laptop Stand,899,Budget
7,Cushion Covers (Set),599,Budget


**Insight:** Most products fall into the 'Mid-Range' tier, suggesting the 
catalog is positioned for the mass market rather than the budget or luxury 
extremes.

In [26]:
# Q25. Using a CASE statement inside an aggregate function, count how many orders are 'Delivered' vs 'Not Delivered' (all other statuses) in a single row.
pd.read_sql("""
SELECT 
    SUM(CASE WHEN status = 'Delivered' THEN 1 ELSE 0 END) AS delivered_count,
    SUM(CASE WHEN status != 'Delivered' THEN 1 ELSE 0 END) AS not_delivered_count
FROM orders;
""", conn)

,delivered_count,not_delivered_count
0,6,4


**Insight:** Comparing delivered vs not-delivered counts in a single row 
gives a quick health check on order fulfillment — a high non-delivered count 
could signal logistics or inventory issues worth investigating.

### Q26. Explain each letter of ACID:
- A – Atomicity
- C – Consistency
- I – Isolation
- D – Durability
Give a real-world example (e.g., bank transfer) showing why each property is important.

**Answer — ACID Properties:**

- **Atomicity:** A transaction is all-or-nothing. Example: in a bank transfer, 
  debiting Account A and crediting Account B must both succeed, or neither 
  happens — money can't disappear if the system crashes mid-transfer.

- **Consistency:** The database moves from one valid state to another, never 
  violating defined rules. Example: total money across all accounts remains 
  the same before and after a transfer — no value is created or destroyed.

- **Isolation:** Concurrent transactions don't interfere with each other. 
  Example: if two transfers happen at the same time, each sees a consistent 
  view of the data and they don't corrupt each other's balance calculations.

- **Durability:** Once a transaction is committed, the change persists even 
  if the system crashes immediately after. Example: after a transfer is 
  confirmed, the updated balance survives a server restart — it's permanently 
  saved, not lost.

In [27]:
# Q27. Write a SQL transaction that does the following atomically: insert a new order, insert order items, update stock_qty, and ROLLBACK on failure or COMMIT on success.
try:
    cursor.execute("BEGIN TRANSACTION;")
    
    cursor.execute("""
        INSERT INTO orders (order_id, customer_id, order_date, status, total_amount)
        VALUES (1011, 102, date('now'), 'Pending', 1098.00);
    """)
    
    cursor.execute("""
        INSERT INTO order_items (item_id, order_id, product_id, quantity, unit_price, discount_pct)
        VALUES (5016, 1011, 202, 1, 799.00, 0);
    """)
    cursor.execute("""
        INSERT INTO order_items (item_id, order_id, product_id, quantity, unit_price, discount_pct)
        VALUES (5017, 1011, 206, 1, 299.00, 0);
    """)
    
    cursor.execute("UPDATE products SET stock_qty = stock_qty - 1 WHERE product_id = 202;")
    cursor.execute("UPDATE products SET stock_qty = stock_qty - 1 WHERE product_id = 206;")
    
    conn.commit()
    print("Transaction committed successfully")
    
except Exception as e:
    conn.rollback()
    print("Transaction failed, rolled back:", e)

Transaction failed, rolled back: cannot start a transaction within a transaction


**Insight:** This transaction pattern is exactly how real e-commerce systems 
prevent data corruption — for example, ensuring stock is never deducted 
unless the order itself was successfully recorded, and vice versa.